In [21]:
import os
from dotenv import load_dotenv
from openai import AsyncOpenAI
from agents import Agent, Runner, trace, function_tool, OpenAIChatCompletionsModel, output_guardrail, GuardrailFunctionOutput
from pydantic import BaseModel, Field
from sendgrid import SendGridAPIClient
from sendgrid.helpers.mail import Mail
sg = SendGridAPIClient(os.environ.get('SENDGRID_API_KEY'))

In [5]:
load_dotenv(override=True)

True

In [6]:
openai_api_key = os.getenv('OPENAI_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

if  openrouter_api_key:
    print(f"OpenRouter API Key exists and begins {openrouter_api_key[:6]}")
else:
    print("OpenRouter API Key not set (and this is optional)")

if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:4]}")
else:
    print("Groq API Key not set (and this is optional)")

OpenAI API Key exists and begins sk-proj-
Google API Key exists and begins AI
OpenRouter API Key exists and begins sk-or-
Groq API Key exists and begins gsk_


In [41]:
intro = """ You are a career strategist and student outreach agent. Your goal is to draft email according to this ---
"""

instructions1 = intro + "Your email style is very bad, goes out of context and tries to ask some insider information"
instructions2 = intro + "Your email style is witty, engaging, and humorous and and make fun of how irrelavntly they have put the job description, suggest them to change it."
instructions3 = intro + "Your email style is punchy, high-impact, and unapologetically confident—framing me as a top-tier candidate who brings unmatched value to the team."

In [42]:
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
GROQ_BASE_URL = "https://api.groq.com/openai/v1"

In [43]:
gemini_client = AsyncOpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)
openrouter_client = AsyncOpenAI(base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key)
groq_client = AsyncOpenAI(base_url=GROQ_BASE_URL, api_key=groq_api_key)

In [44]:
gemini_model = OpenAIChatCompletionsModel(model="gemini-3.1-flash-lite", openai_client=gemini_client)
kimi_model = OpenAIChatCompletionsModel(model="moonshotai/kimi-k3", openai_client=openrouter_client)
oss_model = OpenAIChatCompletionsModel(model="openai/gpt-oss-120b", openai_client=groq_client)

In [45]:
student_agent1 = Agent(name="Gemini Student Agent", instructions=instructions1, model=gemini_model)
student_agent2 = Agent(name="Kimi-k3 Student Agent", instructions=instructions2, model=kimi_model)
student_agent3 = Agent(name="GPT-oss Student Agent", instructions=instructions3, model=oss_model)

In [46]:
description = "Use this tool to write a reach out email.In the input instruct it to write a student mail based on job descriptiona and user input"

tool1 = student_agent1.as_tool(tool_name="studnet_agent_1", tool_description=description)
tool2 = student_agent2.as_tool(tool_name="studnet_agent_2", tool_description=description)
tool3 = student_agent3.as_tool(tool_name="studnet_agent_3", tool_description=description)

In [47]:
job_description = """Sierra is a pioneering platform dedicated to transforming customer interactions through advanced AI agents. Serving some of the world's most prominent brands, including The GAP, Rocket Mortgage, SoFi, Sutter Health, and SoftBank, Sierra specializes in creating intelligent solutions that enhance customer service and drive business growth. The company operates primarily from its headquarters in San Francisco and has expanding offices across North America, Europe, and Asia, reflecting its global reach and commitment to innovation.

Guided by core values such as Trust, Customer Obsession, Craftsmanship, Intensity, and a commitment to work-life balance, Sierra fosters a culture of excellence, integrity, and collaboration. Its leadership team includes industry veterans like Bret Taylor, Board Chair of OpenAI and former co-CEO of Salesforce, and Clay Bavor, a veteran at Google leading initiatives in AR/VR and productivity tools. These leaders drive Sierra's vision to develop cutting-edge AI solutions that redefine customer engagement across various industries.

At Sierra, the focus is on building scalable, reliable, and impactful AI agents that empower organizations to deliver personalized, efficient, and memorable customer experiences. The company's innovative approach combines deep technological expertise with a customer-centric mindset, ensuring that its solutions are not only powerful but also aligned with client needs and industry standards.

About The Role

Sierra is seeking a talented and motivated AI Engineer to join its Tech, Media & Telecom team. This role offers a unique opportunity to work at the forefront of AI development, building production-grade AI agents that serve as critical components in enhancing customer interactions for some of the world’s leading brands. The successful candidate will be instrumental in designing, developing, and deploying scalable AI systems that directly contribute to Sierra’s growth and innovation objectives.

In this position, you will have ownership over the entire Agent Development Life Cycle, from initial pilot phases to full deployment and ongoing refinement. You will collaborate closely with enterprise clients and startups alike, translating complex business challenges into effective AI solutions. Additionally, your insights and work will influence the evolution of Sierra’s core platform, helping to identify unmet needs, prototype new features, and shape the future of AI agent technology.

This role requires a proactive, creative, and technically skilled individual who thrives in dynamic environments and is passionate about leveraging AI to solve real-world problems. You will be part of a collaborative team that values craftsmanship, innovation, and customer focus, contributing to projects that impact millions of users worldwide.

Qualifications

To excel in this role, candidates should possess a strong background in building and scaling end-to-end production systems. Demonstrated experience with AI, machine learning, or related fields is essential, along with proficiency in relevant programming languages such as Python, React, TypeScript, or Go. The ideal candidate will have excellent problem-solving skills, especially in fast-paced, ambiguous environments, and a strong desire to innovate and experiment.

Effective communication skills are critical, as you will frequently interact with both technical and non-technical stakeholders. A builder’s mindset, high agency, and the ability to work independently are highly valued. Previous experience in deploying AI/LLM systems in production environments, working with enterprise clients, or leading technical projects will be advantageous.

Additional preferred qualifications include familiarity with tools such as eval frameworks, agent tooling, RAG pipelines, prompt engineering, and experience with voice or conversational AI. Prior entrepreneurial experience or roles involving customer engagement and stakeholder management are also considered a plus.

Responsibilities

Design and deliver production-grade AI agents that are reliable, scalable, and intuitive, directly contributing to Sierra’s growth and customer success. You will be responsible for building high-performance systems that handle complex customer interactions across various industries such as finance, healthcare, and commerce.

Lead the entire Agent Development Life Cycle, from initial pilot testing to deployment, tuning, and continuous iteration. Define and implement best practices for development, deployment, and maintenance of AI agents, ensuring quality and efficiency at every stage.

Collaborate with large enterprises and innovative startups to understand their unique business challenges and craft tailored AI solutions. Your work will help transform customer engagement strategies and operational workflows at scale.

Contribute to the evolution of Sierra’s core platform by surfacing unmet needs, prototyping new features, and working cross-functionally with research, product, and platform teams. Your insights will help shape the future of Sierra’s AI agent capabilities and overall product roadmap.

Engage in projects such as developing goal-oriented agents, personalization systems powered by Sierra’s Context Engine, conversational discovery tools, and deep troubleshooting agents for telecommunications and connected devices. Experiment with voice experiences and tailored Voice Personas to enhance brand-specific customer interactions.

Benefits

Sierra offers a comprehensive benefits package designed to support the well-being and growth of its employees. Full-time team members enjoy flexible, unlimited paid time off, enabling work-life balance. The company provides medical, dental, and vision coverage for employees and their families, along with life insurance and disability benefits.

Retirement plans are available depending on the country of employment, complemented by parental leave policies and fertility and family-building benefits through partners like Carrot. Employees also receive daily meals, snacks, and coffee to foster a productive environment, along with a discretionary benefit stipend to spend on personal needs or interests.

Additional perks include opportunities for professional development and community engagement, all within a culture that emphasizes respect, support, and recognition of individual achievements. Sierra’s benefits are aligned with its core values and adaptable to regional requirements, with participation in equity plans available for eligible employees
"""

In [48]:
user = """I'm Kavya Hosamane Jayanna, a Full Stack AI Engineer (RAG · LLM · .NET · React · AWS) with an MS in Computer Science from Binghamton University and a BE in Electronics and Communication from MVJ College of Engineering, Bangalore. I'm AWS Certified Cloud Practitioner. My background includes 2+ years as a Software/Senior Software Engineer at Happiest Minds Technologies, plus research assistant roles in Generative AI, multimodal research, and a Product Design & Data Analytics internship at Neuro20 Technologies. My skills span Python, .NET, React, Machine Learning, Deep Learning (PyTorch, TensorFlow, CNNs), LangChain, Vector Embeddings/RAG systems, Hugging Face, AWS, and SQL.

LinkedIn: https://linkedin.com/in/kavya-h-j
GitHub: https://github.com/kavyahj04
Portfolio: https://my-porttfolio-lemon.vercel.app/
"""

In [49]:
user_input = "Hi I want to apply to this job---" +"\n" + job_description + "\n" + "This is my information -----   " + user + "\n" + "Please help me to generate a mail"

In [50]:
sys_instructions = """You are an email sales manager. Do NOT write the email yourself.
1. Call student_agent_1, student_agent_2 and student_agent_3, each with the full job description and candidate info.
2. Compare the three drafts and pick the best one.
3. Call send_email_tool exactly once with the chosen email.
"""

In [51]:
def send_email(message):
    try:
        response = sg.send(message)
        print(f"Success! Status Code: {response.status_code}")
    except Exception as e:
        print(f"An error occurred: {e}")

In [52]:
@function_tool
def send_email_tool(subject: str, html_body: str) -> str:
    """Send an email to the recipient with the given subject and HTML body.

    Args:
        subject: The subject line of the email
        html_body: The body of the email as HTML
    """
    message = Mail(
        from_email='kavyahj26@gmail.com',  # Must match your verified sender/domain
        to_emails='khj@binghamton.edu',
        subject=subject,
        html_content=html_body,
    )
    send_email(message)
    return "Email sent successfully"


In [53]:
tools = [tool1, tool2, tool3, send_email_tool]

In [54]:
email_sender = Agent(name="Student helper", instructions=sys_instructions, model="gpt-4o-mini", tools=tools)

In [55]:
with trace("Student Helper Agent"):
    results = await Runner.run(email_sender, user_input)
    print(results.final_output)

Success! Status Code: 202
Your application email for the AI Engineer position at Sierra has been successfully sent! If you need any further assistance, feel free to ask. Good luck!


## Structured Outputs

In [56]:
class EmailReview(BaseModel):
    is_professional: bool = Field(description="Whether the email is professional and appropriate")
    number_of_sentences: int = Field(description="The number of sentences in the body of the email, not including the greeting and signature")
    contains_placeholders: bool = Field(description="Whether the email contains placeholders for personalization")

In [57]:
EmailReview.model_json_schema()

{'properties': {'is_professional': {'description': 'Whether the email is professional and appropriate',
   'title': 'Is Professional',
   'type': 'boolean'},
  'number_of_sentences': {'description': 'The number of sentences in the body of the email, not including the greeting and signature',
   'title': 'Number Of Sentences',
   'type': 'integer'},
  'contains_placeholders': {'description': 'Whether the email contains placeholders for personalization',
   'title': 'Contains Placeholders',
   'type': 'boolean'}},
 'required': ['is_professional',
  'number_of_sentences',
  'contains_placeholders'],
 'title': 'EmailReview',
 'type': 'object'}

In [58]:
email = """
Hi [first_name],

I'm hitting you up to see if you'd like to buy our product. It's really great. You'll miss out if you don't buy it.

Laters.

Ed
"""

In [59]:
checker = Agent(name = "checker", instructions="You review student reach out emails", model = "gpt-5.4-mini", output_type=EmailReview)

In [60]:
result = await Runner.run(checker, email)

In [62]:
review = result.final_output
review

EmailReview(is_professional=False, number_of_sentences=3, contains_placeholders=True)